In [1]:
pip install qiskit qiskit-aer qiskit-machine-learning qiskit-algorithms numpy matplotlib scikit-learn torch


   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   ---------------------------------------- 0/2 [qiskit-machine-learning]
   -------------------- ------------------- 1/2 [qiskit-algorithms]
   -------------------- ------------------- 1/2 [qiskit-algorithms]
   -------------------- ------------------- 1/2 [qiskit-algorithms]
   -------------------- ------------------- 1/2 [qiskit


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install --upgrade qiskit qiskit-machine-learning qiskit-algorithms numpy matplotlib

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.8 MB 4.9 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.8 MB 4.4 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.8 MB 4.2 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/9.8 MB 4.1 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.8 MB 4.1 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.8 MB 4.0 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.8 MB 4.0 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.8 MB 4.0 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.8 MB 4.0 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.8 MB 4.0 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.8 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 3.9 MB/s  0:00:02
  Attempting uninstall


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from qiskit_aer import Aer
from qiskit.primitives import StatevectorSampler 
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_algorithms.optimizers import COBYLA
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Optimize1qGatesDecomposition, CommutativeCancellation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

In [4]:
X, y = make_classification(n_samples=100,      # Total number of features
                           n_features=2,
                           n_informative=2,    # Ensure sum ≤ total features
                           n_redundant=0,      # No redundant features
                           n_clusters_per_class=1,
                           random_state=42)

In [5]:
# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [7]:
feature_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='linear')

C:\Users\SimantaRajbangshi\AppData\Local\Temp\ipykernel_26180\2133380198.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='linear')


In [8]:
ansatz = RealAmplitudes(num_qubits=2, reps=3, entanglement='linear')

C:\Users\SimantaRajbangshi\AppData\Local\Temp\ipykernel_26180\2662761407.py:1: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(num_qubits=2, reps=3, entanglement='linear')


In [9]:
sampler = StatevectorSampler()

In [10]:
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=sampler)

In [11]:
optimizer = COBYLA(maxiter=100)  # Ensure optimizer is an instance, not a string

In [12]:
pass_manager = PassManager([
    Optimize1qGatesDecomposition(basis=['u3', 'cx']),
    CommutativeCancellation()
])

In [13]:
vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    sampler=sampler,
    pass_manager=pass_manager  # Provide pass manager to avoid transpilation warnings
)

In [14]:
vqc.fit(X_train, y_train)

In [15]:
y_pred = vqc.predict(X_test)

# Calculate accuracy
accuracy = np.mean(y_pred == y_test)
print(f"VQC Accuracy: {accuracy * 100:.2f}%")

VQC Accuracy: 45.00%


In [16]:
import numpy as np
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.circuit.library import ZZFeatureMap
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons

In [17]:
# Step 1: Custom Fidelity Function for Quantum Kernel
def custom_fidelity(sampler, circuit_1, circuit_2):
    """
    Compute the fidelity between two quantum states using the StatevectorSampler primitive.
    """
    combined_circuit = circuit_1.compose(circuit_2.inverse())  # Combine the circuits
    job = sampler.run([combined_circuit])
    result = job.result()
    probabilities = result.quasi_dists[0]
    # Fidelity is the probability of measuring the all-zero state
    num_qubits = combined_circuit.num_qubits
    zero_state = tuple([0] * num_qubits)
    fidelity = probabilities.get(zero_state)
    return fidelity

In [18]:
X, y = make_moons(n_samples=100, noise=0.1, random_state=42)

In [19]:
# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [20]:
feature_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='linear')

C:\Users\SimantaRajbangshi\AppData\Local\Temp\ipykernel_26180\2133380198.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='linear')


In [21]:
sampler = StatevectorSampler()

In [22]:
class CustomFidelityQuantumKernel(FidelityQuantumKernel):
    def __init__(self, feature_map, sampler):
        super().__init__(feature_map=feature_map, fidelity=None)
        self.sampler = sampler

    def _compute_fidelity(self, circuit_1, circuit_2):
        return custom_fidelity(self.sampler, circuit_1, circuit_2)

In [23]:
quantum_kernel = CustomFidelityQuantumKernel(feature_map=feature_map, sampler=sampler)

X_train_kernel = quantum_kernel.evaluate(X_train)
X_test_kernel = quantum_kernel.evaluate(X_test, X_train)

In [24]:
svm = SVC(kernel="precomputed")
svm.fit(X_train_kernel, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'precomputed'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [25]:
y_pred = svm.predict(X_test_kernel)

accuracy = np.mean(y_pred == y_test)
print(f"Quantum Kernel SVM Accuracy: {accuracy * 100:.2f}%")

Quantum Kernel SVM Accuracy: 60.00%


In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import RealAmplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

In [27]:
X, y = make_moons(n_samples=100, noise=0.1, random_state=42)

In [28]:
# Normalize and split data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [29]:
# Convert to PyTorch tensors
X_train_torch = torch.tensor(X_train, dtype=torch.float32)
X_test_torch = torch.tensor(X_test, dtype=torch.float32)
y_train_torch = torch.tensor(y_train, dtype=torch.long)
y_test_torch = torch.tensor(y_test, dtype=torch.long)

In [30]:
num_qubits = 2
quantum_circuit = RealAmplitudes(num_qubits, reps=3)

C:\Users\SimantaRajbangshi\AppData\Local\Temp\ipykernel_26180\1101268564.py:2: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  quantum_circuit = RealAmplitudes(num_qubits, reps=3)


In [31]:
estimator = StatevectorEstimator()
qnn = EstimatorQNN(
    circuit=quantum_circuit,
    estimator=estimator,
    input_params=quantum_circuit.parameters[:2],
    weight_params=quantum_circuit.parameters[2:]
)

In [32]:
quantum_layer = TorchConnector(qnn)

In [33]:
class HybridQuantumNN(nn.Module):
    def __init__(self):
        super(HybridQuantumNN, self).__init__()
        self.fc1 = nn.Linear(2, 2)
        self.quantum = quantum_layer
        self.fc2 = nn.Linear(1, 2)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = self.quantum(x)
        x = self.fc2(x)
        return x

In [34]:
model = HybridQuantumNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [35]:
epochs = 50
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_torch)
    loss = criterion(outputs, y_train_torch)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 10/50, Loss: 0.7103
Epoch 20/50, Loss: 0.6944
Epoch 30/50, Loss: 0.6953
Epoch 40/50, Loss: 0.6887
Epoch 50/50, Loss: 0.6836


In [36]:
with torch.no_grad():
    y_pred = model(X_test_torch).argmax(dim=1)
    accuracy = (y_pred == y_test_torch).float().mean()
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 45.00%
